In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

df_adult = pd.read_csv('adult.csv')
#df_adult = df_adult.sample(5000, random_state=42)
df_adult['income'] = df_adult['income'].apply(lambda x: 1 if '>50K' in str(x) else 0)

X_pd = pd.get_dummies(df_adult.drop('income', axis=1), drop_first=True).astype(float)
y = df_adult['income'].values

X_train, X_test, y_train, y_test = train_test_split(X_pd, y, test_size=0.3, random_state=42)

X_train_np = X_train.values
col_names = list(X_train.columns)

X_base = np.ones((len(y_train), 1))
base_model = sm.Logit(y_train, X_base).fit(disp=0, method='lbfgs')
print(f"Startowe AIC: {base_model.aic:.2f}\n")

features_to_check = list(range(len(col_names)))
selected_idx = []
current_aic = base_model.aic 

while features_to_check:
    best_aic = current_aic
    best_idx = None
    
    for idx in features_to_check:
        test_indices = selected_idx + [idx]
        X_test_sm = sm.add_constant(X_train_np[:, test_indices])
        
        model_sm = sm.Logit(y_train, X_test_sm).fit(disp=0, method='lbfgs') 
        
        if model_sm.aic < best_aic:
            best_aic = model_sm.aic
            best_idx = idx
            
    if best_idx is not None:
        selected_idx.append(best_idx)
        features_to_check.remove(best_idx)
        current_aic = best_aic
        print(f"Wybrano: {col_names[best_idx]:<30} | AIC: {best_aic:.4f}")
    else:
        break

selected_features = [col_names[i] for i in selected_idx]
X_train_final = sm.add_constant(X_train[selected_features])
X_test_final = sm.add_constant(X_test[selected_features])

final_model = sm.Logit(y_train, X_train_final).fit(disp=0, method='lbfgs')
print(final_model.summary())

y_pred = (final_model.predict(X_test_final) >= 0.5).astype(int)
print(f"\nDokładność (Accuracy) na zbiorze testowym: {accuracy_score(y_test, y_pred):.4f}")